# GLM de primer nivel — movimiento de la mano derecha (46 sujetos)
Modelo por sujeto sobre los BOLD ya preprocesados (MNI 2 mm, suavizados 6 mm):
`move` vs. línea base (`rest`), HRF canónica SPM, filtro pasa-altas 128 s, ruido AR(1).
**No se vuelve a suavizar**: el suavizado de 6 mm ya se hizo en el preprocesamiento.
Salidas por sujeto en `derivatives_unidad3/glm_primer_nivel/<sub>/`: mapa **z**, mapa de **efecto** (% de cambio de señal), matriz de diseño y un PNG de revisión.
Es reanudable: si el mapa z de un sujeto existe, se salta.

## 0. Dependencias

In [2]:
# Descomenta y ejecuta UNA vez; luego reinicia el kernel
%pip install nilearn
import nilearn
print("nilearn", nilearn.__version__)

  Using cached nilearn-0.14.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached markupsafe-3.0.4-cp311-cp311-win_amd64.whl.metadata (2.8 kB)
Using cached nilearn-0.14.1-py3-none-any.whl (11.5 MB)
Using cached jinja2-3.1.6-py3-none-any.whl (134 kB)
Using cached markupsafe-3.0.4-cp311-cp311-win_amd64.whl (14 kB)
Note: you may need to restart the kernel to use updated packages.
nilearn 0.14.1



[notice] A new release of pip is available: 24.0 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Rutas y parámetros

In [3]:
import gc, time, traceback
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
from nilearn.glm.first_level import FirstLevelModel
from nilearn.masking import compute_epi_mask
from nilearn import plotting

RAIZ        = Path.cwd().resolve()                  # el notebook esta en Proyecto_U3
CSV_SUJETOS = RAIZ / "sujetos_46.csv"
DATA_DIR    = RAIZ / "ds005366_subset"              # de aqui salen los events.tsv
OUT_DIR     = RAIZ / "derivatives_unidad3"          # preprocesados
GLM_DIR     = OUT_DIR / "glm_primer_nivel"
MNI_PATH    = Path(r"C:\fMRI\Worskshop-fMRI-preprocessing-basics\data\external\templates\MNI152_T1_2mm.nii.gz")

HIGH_PASS_HZ = 1 / 128      # filtro pasa-altas (bloques de 30 s => se conserva la tarea)
USAR_FD      = True         # FD como regresor de ruido (util en pacientes/ninos con mas movimiento)
Z_UMBRAL     = 3.1          # solo para el PNG de revision

for n, p in [("CSV", CSV_SUJETOS), ("DATA", DATA_DIR), ("OUT", OUT_DIR), ("MNI", MNI_PATH)]:
    print(f"{n:5s} {'OK ' if p.exists() else 'NO EXISTE'} {p}")
GLM_DIR.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(CSV_SUJETOS)
df.groupby(["grupo", "sex"]).size()

CSV   OK  C:\Users\Lenovo\Documents\Paulina 9S\P Imagenes\Proyecto_U3\sujetos_46.csv
DATA  OK  C:\Users\Lenovo\Documents\Paulina 9S\P Imagenes\Proyecto_U3\ds005366_subset
OUT   OK  C:\Users\Lenovo\Documents\Paulina 9S\P Imagenes\Proyecto_U3\derivatives_unidad3
MNI   OK  C:\fMRI\Worskshop-fMRI-preprocessing-basics\data\external\templates\MNI152_T1_2mm.nii.gz


grupo      sex
epilepsia  F       8
           M       8
sano       F      15
           M      15
dtype: int64

## 2. Función de primer nivel
Un sujeto a la vez, con máscara cerebral calculada de su propia media funcional en MNI. Solo la condición `move` entra al modelo; `rest` queda como línea base implícita.

In [4]:
def primer_nivel(r):
    sub = r["sub"]
    nombre = f"{sub}_ses-{r['ses']}_task-Motor2Class_run-{int(r['run'])}"
    d_pre = OUT_DIR / sub
    d_glm = GLM_DIR / sub
    d_glm.mkdir(parents=True, exist_ok=True)
    f_z = d_glm / f"{sub}_contrast-move_stat-z.nii.gz"
    if f_z.exists():
        print(f"[{sub}] ya procesado, se salta")
        return "ya existia"

    bold = d_pre / f"{nombre}_space-MNI152_desc-preproc_bold.nii.gz"
    media = d_pre / f"{nombre}_mean_mni.nii.gz"
    ev_path = DATA_DIR / sub / f"ses-{r['ses']}" / r["events_file"]

    # eventos: solo 'move' (rest = linea base)
    ev = pd.read_csv(ev_path, sep="\t")
    ev = ev[ev["trial_type"] == "move"][["onset", "duration", "trial_type"]].reset_index(drop=True)

    nvol = nib.load(str(bold)).shape[3]
    tr = float(r["TR"])

    # confusores: FD (si coincide con el numero de volumenes)
    confs = None
    mot = d_pre / "motion.csv"
    if USAR_FD and mot.exists():
        fd = pd.read_csv(mot)["FD"].fillna(0).to_numpy(float)
        if len(fd) == nvol:
            confs = pd.DataFrame({"FD": fd})
        else:
            print(f"   [aviso] FD tiene {len(fd)} valores y hay {nvol} volumenes: se omite FD")

    mascara = compute_epi_mask(str(media))
    glm = FirstLevelModel(
        t_r=tr, hrf_model="spm", drift_model="cosine", high_pass=HIGH_PASS_HZ,
        noise_model="ar1", smoothing_fwhm=None, mask_img=mascara,
        minimize_memory=True, n_jobs=1, verbose=0,
    )
    glm.fit(str(bold), events=ev, confounds=confs)

    res = glm.compute_contrast("move", output_type="all")
    res["z_score"].to_filename(str(f_z))
    res["effect_size"].to_filename(str(d_glm / f"{sub}_contrast-move_stat-effect.nii.gz"))
    res["effect_variance"].to_filename(str(d_glm / f"{sub}_contrast-move_stat-variance.nii.gz"))
    dm = glm.design_matrices_[0]
    dm.to_csv(d_glm / f"{sub}_design_matrix.csv", index=False)

    # PNG de revision: mapa z sobre la plantilla
    fig = plt.figure(figsize=(11, 3.2))
    plotting.plot_stat_map(res["z_score"], bg_img=str(MNI_PATH), threshold=Z_UMBRAL,
                           display_mode="z", cut_coords=[46, 52, 58, 64], figure=fig,
                           title=f"{sub} ({r['grupo']}) move > rest, z>{Z_UMBRAL}")
    fig.savefig(d_glm / f"{sub}_zmap.png", dpi=100)
    plt.close(fig)

    zmax = float(np.nanmax(res["z_score"].get_fdata()))
    del glm, res, mascara, ev, confs
    gc.collect()
    return f"ok (z max = {zmax:.1f})"

## 3. Prueba con un sano y un paciente

In [1]:
def correr(subs):
    out = []
    for _, r in df[df["sub"].isin(subs)].iterrows():
        t0 = time.time()
        try:
            msg = primer_nivel(r)
            out.append(dict(sub=r["sub"], grupo=r["grupo"], estado=msg, minutos=round((time.time() - t0) / 60, 1)))
        except Exception as e:
            traceback.print_exc()
            out.append(dict(sub=r["sub"], grupo=r["grupo"], estado=f"ERROR {e!r}", minutos=round((time.time() - t0) / 60, 1)))
        finally:
            gc.collect()
    return out

prueba = [df[df.grupo == "sano"]["sub"].iloc[0], df[df.grupo == "epilepsia"]["sub"].iloc[0]]
pd.DataFrame(correr(prueba))

NameError: name 'df' is not defined

Abre los `*_zmap.png` de estos dos sujetos en `glm_primer_nivel/<sub>/`: deberías ver activación en la región central contralateral (hemisferio **izquierdo**, giro precentral/postcentral) para la mano derecha. Si se ve bien, corre el lote.

## 4. Lote completo

In [ ]:
log = []
for s in df["sub"]:
    log += correr([s])
    pd.DataFrame(log).to_csv(GLM_DIR / "glm_log.csv", index=False)
res_glm = pd.DataFrame(log)
print(res_glm.estado.str.split(" ").str[0].value_counts())
res_glm[res_glm.estado.str.startswith("ERROR")]

## 5. Resumen: z máximo por sujeto

In [ ]:
filas = []
for _, r in df.iterrows():
    f = GLM_DIR / r["sub"] / f"{r['sub']}_contrast-move_stat-z.nii.gz"
    if f.exists():
        z = nib.load(str(f)).get_fdata()
        filas.append(dict(sub=r["sub"], grupo=r["grupo"], z_max=np.nanmax(z), voxeles_z_gt_3_1=int((z > 3.1).sum())))
resumen = pd.DataFrame(filas)
resumen.to_csv(GLM_DIR / "resumen_primer_nivel.csv", index=False)
print(resumen.groupby("grupo")[["z_max", "voxeles_z_gt_3_1"]].describe().T)
resumen.sort_values("z_max").head(8)   # sujetos con menor respuesta: revisalos